# Preliminary Machine Learning Direction -  London Airbnb

### Project Goal

The goal of this project is to understand which **listing and host characteristics are associated with a top Airbnb rating (4.8 or higher)** and use the findings from the complete exploratory data analysis to define an appropriate machine-learning direction.

This notebook is a **planning and handoff stage**, not the final machine-learning analysis. It brings together findings from:

**Data Cleaning → Univariate EDA → Bivariate EDA → Multivariate EDA → Preliminary ML Direction**

### Proposed ML Task

**Binary Classification**

The model will predict whether a rated London Airbnb listing belongs to one of two classes:

- **1 - Top-Rated:** `review_scores_rating >= 4.8`
- **0 - Not Top-Rated:** `review_scores_rating < 4.8`

The modeling target is therefore:

`top_rated`

Only listings with an observed review rating are eligible for this target-based modeling task.

### Why Classification?

The project question asks whether a listing is **top-rated or not top-rated**, which produces a binary outcome rather than a continuous value.

The EDA also identified several listing, host, location, and booking characteristics that may provide predictive information about this outcome.

The purpose of the future model will be to evaluate how well these characteristics collectively distinguish top-rated from non-top-rated listings.

> **Important:** Predictive relationships will be interpreted as associations, not evidence that a feature causes a listing to receive a higher rating.

## Machine Learning Workflow

The preliminary modeling plan follows the decisions made during EDA:

**1. Define Target**  
`top_rated` → Top-Rated vs Not Top-Rated

↓  

**2. Select Modeling Population**  
Use only listings with an observed rating

↓  

**3. Select Candidate Features**  
Carry forward useful listing, host, location, and booking features identified during EDA

↓  

**4. Remove Leakage / Low-Information Features**  
Exclude variables that reveal the rating outcome or provide little useful predictive information

↓  

**5. Prepare Features**  
Handle numeric transformations, categorical encoding, missing-value indicators, and nonlinear relationships where appropriate

↓  

**6. Establish a Baseline Model**  
Begin with an interpretable binary-classification baseline

↓  

**7. Compare with Nonlinear Models**  
Evaluate whether more flexible models capture relationships identified during EDA

↓  

**8. Evaluate Models**  
Compare validation performance using appropriate classification metrics

↓  

**9. Interpret Important Features**  
Determine which features contribute useful predictive information

↓  

**10. Final Model Decision**  
Select the model and feature set based on validation performance, interpretability, and project goals

### Preliminary ML Workflow

![Preliminary Machine Learning Workflow](../figures/fig_ml_workflow.png)

**Figure. Preliminary machine-learning workflow for the London Airbnb project.**  
The workflow carries the decisions from EDA into feature preparation, classification-model comparison, evaluation, interpretation, and eventual model selection. At this stage, it represents the proposed modeling direction rather than final model results.


## EDA → Machine Learning Handoff

The preliminary machine-learning direction is based on decisions made across the complete exploratory analysis rather than on a single visualization.

The EDA identified several listing, host, location, and booking characteristics that may help distinguish top-rated from non-top-rated listings. It also identified variables that should be transformed, treated cautiously, or excluded because of leakage or limited information.

### Main EDA Findings Informing ML

- **Target:** `top_rated` is a binary outcome, making this a classification problem.
- **Target balance:** Among rated listings, approximately **56.1% are top-rated** and **43.9% are not top-rated**, so severe class imbalance was not observed.
- **Price:** Top-rated listings show some price separation, but the effect is small and price is strongly right-skewed. A log transformation should therefore be considered.
- **Room type:** Room type shows some association with top-rated status and should be retained as a categorical feature.
- **Listing capacity:** `accommodates` shows a nonlinear relationship with top-rated status and should not automatically be treated as a simple linear effect.
- **Location:** Top-rated rates vary across London boroughs, indicating that location may provide useful predictive context.
- **Host type:** Single-listing and multi-listing hosts show one of the strongest categorical differences identified during bivariate EDA.
- **Host tenure:** Host experience shows only a weak, non-monotonic relationship and should be retained only if it improves validation performance.
- **Amenity count:** Higher amenity counts show some predictive signal, while using thousands of individual free-text amenities would require additional feature engineering.
- **Minimum nights:** Minimum-stay rules show a meaningful but nonlinear relationship with top-rated status.
- **Feature redundancy:** Listing capacity and price are strongly related, so their combined contribution should be checked during model validation.
- **Leakage:** Review sub-scores and `host_is_superhost` should not be used as predictors because they may reveal information closely related to the rating outcome.
- **Low-information variables:** Features that are nearly constant or provide little variation should not be prioritized simply because they are available.

These findings define the initial candidate-feature set and preprocessing strategy. Final feature inclusion will be determined using validation performance rather than bivariate statistical significance alone.

In [3]:
import pandas as pd
import numpy as np

from IPython.display import display

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [4]:
candidate_features = pd.DataFrame([
    ["Price", "price_capped_imp + price missing flag",
     "Numeric", "Retain",
     "Log transformation; preserve missing-price information"],

    ["Room type", "room_type",
     "Categorical", "Retain",
     "Categorical encoding"],

    ["Listing capacity", "accommodates",
     "Numeric", "Retain",
     "Evaluate nonlinear treatment"],

    ["Location", "neighbourhood_cleansed",
     "Categorical", "Retain",
     "Categorical encoding"],

    ["Host type", "calculated_host_listings_count / derived host type",
     "Numeric / derived categorical", "Retain",
     "Strong EDA candidate; compare useful representation"],

    ["Host tenure", "hosts_time_as_host_years",
     "Numeric", "Validate",
     "Weak nonlinear relationship; retain only if useful"],

    ["Amenity count", "derived amenity_count",
     "Numeric", "Retain",
     "Use count rather than raw free-text amenities initially"],

    ["Minimum nights", "minimum_nights_capped",
     "Numeric", "Retain",
     "Nonlinear relationship; consider grouped/nonlinear treatment"]

], columns=[
    "Feature",
    "Model Variable",
    "Type",
    "EDA Decision",
    "ML Preparation"
])

display(candidate_features)

,Feature,Model Variable,Type,EDA Decision,ML Preparation
0,Price,price_capped_imp + price missing flag,Numeric,Retain,Log transformation; preserve missing-price inf...
1,Room type,room_type,Categorical,Retain,Categorical encoding
2,Listing capacity,accommodates,Numeric,Retain,Evaluate nonlinear treatment
3,Location,neighbourhood_cleansed,Categorical,Retain,Categorical encoding
4,Host type,calculated_host_listings_count / derived host ...,Numeric / derived categorical,Retain,Strong EDA candidate; compare useful represent...
5,Host tenure,hosts_time_as_host_years,Numeric,Validate,Weak nonlinear relationship; retain only if us...
6,Amenity count,derived amenity_count,Numeric,Retain,Use count rather than raw free-text amenities ...
7,Minimum nights,minimum_nights_capped,Numeric,Retain,Nonlinear relationship; consider grouped/nonli...


### Initial Candidate Feature Set

The table above represents the **initial modeling feature set carried forward from EDA**. These features are not automatically guaranteed a place in the final model.

Feature inclusion will be evaluated using validation performance, redundancy, interpretability, and leakage risk.

The initial feature set intentionally combines:

- **Listing characteristics:** price, room type, and capacity
- **Location information:** London borough
- **Host characteristics:** host type and host tenure
- **Listing information:** amenity count
- **Booking rules:** minimum nights

This provides the preliminary model with information from several different aspects of an Airbnb listing rather than relying on a single feature family.

## Leakage and Feature Exclusion Plan

Before model development, variables that could reveal the target, duplicate target information, act mainly as identifiers, or provide very little predictive information must be excluded.

This is especially important because `top_rated` is derived directly from `review_scores_rating`. Allowing rating-related variables into the feature set could produce artificially strong model performance rather than a useful prediction from listing and host characteristics.

### Features to Exclude or Deprioritize

| Feature / Feature Group | Decision | Reason |
|---|---|---|
| `review_scores_rating` | **Exclude** | Directly determines the `top_rated` target |
| Review sub-scores | **Exclude** | Closely related to the overall review score and create target leakage risk |
| `host_is_superhost` | **Exclude** | Superhost status is influenced by review-related performance and may leak outcome-related information |
| `top_rated` | **Target only** | Prediction target; never included in model inputs |
| Listing / host identifiers | **Exclude from prediction** | IDs identify records or entities rather than representing generalizable listing characteristics |
| `host_identity_verified` | **Deprioritize** | Univariate EDA found it to be near-universal and therefore low-information |
| Raw free-text amenities | **Do not use initially** | Very high-dimensional; use derived `amenity_count` for the preliminary model |
| Near-constant / low-information variables | **Deprioritize** | Little variation means limited ability to distinguish the two target classes |

### Leakage Principle

Only information that would reasonably be available **without knowing the listing's final rating outcome** should be used to predict `top_rated`.

This keeps the preliminary model aligned with the project goal: evaluating whether listing, host, location, and booking characteristics contain useful predictive information about top-rated status.

## Feature Preparation Plan

The candidate features identified during EDA contain a mixture of numeric, categorical, skewed, missing, and potentially nonlinear variables. Therefore, the features will not all be passed to the model in the same form.

The preliminary preprocessing strategy is:

| Feature | Planned Preparation | Reason |
|---|---|---|
| Price | Use cleaned/imputed price, retain missing-price indicator, and consider `log1p` transformation | Price is strongly right-skewed and some original prices are missing |
| Room type | Categorical encoding | Represents distinct listing categories rather than an ordered numeric scale |
| Listing capacity (`accommodates`) | Retain as numeric; allow nonlinear treatment where appropriate | EDA did not show a simple linear relationship with top-rated status |
| Borough (`neighbourhood_cleansed`) | Categorical encoding | London boroughs represent location categories with meaningful differences |
| Host type | Use host portfolio information or derived single-/multi-listing representation | Host type showed one of the strongest categorical associations in bivariate EDA |
| Host tenure | Retain numeric initially; compare nonlinear/grouped treatment if needed | Relationship with top-rated status was weak and non-monotonic |
| Amenity count | Use derived numeric count | Provides a compact representation without expanding thousands of raw amenity strings |
| Minimum nights | Use cleaned/capped value; allow nonlinear treatment | EDA showed meaningful but non-monotonic differences across minimum-stay groups |

### Missing Values

Missing information should be handled using the decisions already established during data cleaning rather than automatically deleting listings.

For example, the cleaned price feature can be accompanied by a missing-price indicator so that the model can distinguish between an observed value and a value requiring preprocessing.

### Categorical Features

Categorical variables such as room type and borough will require encoding before they can be used by models that expect numeric inputs. The exact encoding strategy will depend on the model being evaluated.

### Nonlinear Relationships

EDA showed that several predictors do not have simple linear relationships with `top_rated`. Therefore, the modeling stage should not assume that every one-unit increase produces a constant change in the probability of being top-rated.

More flexible models may be useful for capturing these relationships.

### Important Modeling Principle

All preprocessing decisions that learn information from the data should be fitted using the **training data only** and then applied to validation/test data. This prevents information from the evaluation data from leaking into model preparation.

## Proposed Model Strategy

The project will use **binary classification** because the target `top_rated` has two possible outcomes:

- **1 = Top-Rated**
- **0 = Not Top-Rated**

Rather than selecting a complex model immediately, the modeling stage will compare an interpretable baseline with more flexible nonlinear models.

### Model 1 -  Logistic Regression: Baseline

Logistic Regression will be used as the initial baseline model.

**Why start here?**

- It is appropriate for binary classification.
- It provides a simple benchmark for comparison.
- It helps determine whether the EDA-selected features contain useful predictive information.
- Its coefficients can support interpretation after appropriate preprocessing.

However, EDA identified several nonlinear relationships, so Logistic Regression may not capture all patterns without additional feature preparation.

### Model 2 -  Random Forest: Nonlinear Comparison

Random Forest will be considered as a nonlinear tree-based model.

**Why compare it?**

- It can capture nonlinear relationships.
- It can identify interactions between features.
- It does not require every numeric predictor to have a simple linear relationship with the target.
- It is suitable for relationships observed during EDA, particularly for capacity, minimum nights, amenity count, and host tenure.

### Model 3 -  Boosted-Tree Classifier: Additional Candidate

A boosted-tree classifier may also be evaluated during the later modeling stage.

**Why consider it?**

- It can capture complex nonlinear relationships and interactions.
- It provides another comparison against Logistic Regression and Random Forest.
- It may capture combined patterns across listing, host, location, and booking characteristics.

The exact boosted-tree implementation will be selected during final model development based on the project environment and available libraries.

### Model Selection Principle

The goal is **not simply to choose the most complex model**.

The final model will be selected by considering:

**Validation Performance → Generalization → Interpretability → Project Usefulness**

At this preliminary stage, these are proposed candidate models only. **No model results are being claimed yet.**

## Train, Validation, and Test Strategy

To evaluate the proposed classification models fairly, the rated listings will be divided into separate datasets for model development and evaluation.

### Proposed Data Split

**70,708 Rated Listings**

↓

**Training Set — 70%**

Used to fit preprocessing steps and train the candidate models.

↓

**Validation Set — 15%**

Used to compare candidate models, feature representations, and modeling decisions.

↓

**Test Set — 15%**

Held aside for the final unbiased evaluation after the modeling strategy has been selected.

### Stratified Sampling

The target distribution among rated listings is approximately:

- **56.1% Top-Rated**
- **43.9% Not Top-Rated**

This is not a severe class imbalance. However, a **stratified split** will be used so that approximately the same top-rated / not-top-rated proportion is maintained across the training, validation, and test sets.

Therefore, resampling techniques such as oversampling or undersampling are **not initially required**. They would only be considered later if model evaluation reveals a meaningful class-performance problem.

### Preventing Data Leakage

All preprocessing steps that learn information from the data must be fitted using the **training data only**.

This includes:

- Missing-value treatment
- Numeric transformations
- Categorical encoding
- Feature scaling, when required
- Any data-driven feature selection

The fitted preprocessing steps can then be applied to the validation and test sets.

### Important Evaluation Rule

The **validation set** will guide model and feature decisions.

The **test set should remain untouched until the final model has been selected** so that it provides an unbiased estimate of final model performance.

## Preliminary ML Direction — Final Summary

Based on the complete EDA, the proposed machine-learning task is **binary classification**.

### Proposed Task
Predict `top_rated`:

- **1 = Top-Rated** (rating ≥ 4.8)
- **0 = Not Top-Rated** (rating < 4.8)

The model will use only the **70,708 rated listings**. The target is reasonably balanced at approximately **56.1% top-rated** and **43.9% not top-rated**, so resampling is not initially required.

### EDA Justification
EDA identified useful predictive information across several feature groups:

- **Listing:** price, room type, capacity
- **Location:** London borough
- **Host:** host type and host tenure
- **Amenities:** amenity count
- **Booking rules:** minimum nights

EDA also identified important modeling considerations, including nonlinear relationships, skewed price, possible redundancy between price and capacity, and leakage risk from review-related variables and `host_is_superhost`.

### Proposed Modeling Direction

**Logistic Regression**  
→ interpretable baseline

**Random Forest**  
→ capture nonlinear relationships and interactions

**Boosted-Tree Classifier**  
→ additional nonlinear model for later comparison

Models will eventually be compared using appropriate classification metrics such as **F1-score, precision, recall, accuracy, ROC-AUC, and a confusion matrix**.

### Current Status

This notebook defines the **preliminary ML direction only**.

No final model has been selected and no model-performance results are reported at this stage. Final preprocessing, feature selection, model training, validation, and interpretation will be completed in the next phase of the project.